# Notebook 2: Features, Naive Bayes baseline, and error analysis (Milestone 2)
Group Theta (seed 49). Builds Bag of Words and TF-IDF features on the training part only, compares a majority-class baseline with Multinomial Naive Bayes on the 20% test set, reports the top informative words, plots a 2-D view of the TF-IDF space, and analyzes errors. Saves the fitted models to `models/`.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
from sklearn.decomposition import TruncatedSVD

SEED = 49
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data" / "Reviews.csv"
FIG_DIR = ROOT / "report" / "figures"; FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR = ROOT / "models"; MODEL_DIR.mkdir(parents=True, exist_ok=True)
LABELS = ["negative", "neutral", "positive"]

# Same cleaning as Notebook 1
full = pd.read_csv(DATA, usecols=["Score", "Text", "Summary"])
clean = full.sample(n=50000, random_state=SEED).reset_index(drop=True)
clean = clean.dropna(subset=["Text"]).copy()
clean["Summary"] = clean["Summary"].fillna("")
clean = clean.drop_duplicates(subset=["Text"], keep="first").reset_index(drop=True)
clean["label"] = clean["Score"].apply(lambda s: "negative" if s <= 2 else ("neutral" if s == 3 else "positive"))
clean["doc"] = clean["Summary"] + " " + clean["Text"].astype(str)
print("Rows after cleaning:", len(clean))

In [ ]:
# 80/20 stratified split; every vectorizer and model is fit on the training part only
idx_tr, idx_te = train_test_split(clean.index, test_size=0.2, stratify=clean["label"], random_state=SEED)
train, test = clean.loc[idx_tr], clean.loc[idx_te]
print("Train:", len(train), "| Test:", len(test))

bow = CountVectorizer(max_features=20000, lowercase=True)            # Lesson 6: Bag of Words
X_tr_bow = bow.fit_transform(train["doc"]); X_te_bow = bow.transform(test["doc"])

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=100000, lowercase=True)  # Lesson 6: TF-IDF
X_tr_tfidf = tfidf.fit_transform(train["doc"]); X_te_tfidf = tfidf.transform(test["doc"])

print("Bag of Words vocabulary size:", len(bow.vocabulary_))
print("TF-IDF feature count:", len(tfidf.vocabulary_))

In [ ]:
def evaluate(y_true, y_pred):
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, labels=LABELS, zero_division=0)
    table = pd.DataFrame({"precision (%)": p * 100, "recall (%)": r * 100, "F1 (%)": f * 100}, index=LABELS).round(2)
    cm = pd.DataFrame(confusion_matrix(y_true, y_pred, labels=LABELS), index=[f"true {c}" for c in LABELS],
                      columns=[f"pred {c}" for c in LABELS])
    return accuracy_score(y_true, y_pred) * 100, table, cm

# Majority-class baseline (Lesson 1 framing: a model must beat this to be learning anything)
majority = DummyClassifier(strategy="most_frequent").fit(X_tr_bow, train["label"])
acc_maj, _, _ = evaluate(test["label"], majority.predict(X_te_bow))

# Multinomial Naive Bayes on Bag of Words (Lesson 9). alpha is tiny: estimates by counting, smoothing not in the lectures
nb = MultinomialNB(alpha=1e-9).fit(X_tr_bow, train["label"])
pred_nb = nb.predict(X_te_bow)
acc_nb, table_nb, cm_nb = evaluate(test["label"], pred_nb)

print(f"Majority-class accuracy: {acc_maj:.2f}%")
print(f"Naive Bayes accuracy:    {acc_nb:.2f}%")
print("\nPer-class results (Naive Bayes):")
print(table_nb)
print("\nMacro F1 (%):", round(table_nb["F1 (%)"].mean(), 2))
print("\nConfusion matrix (rows = true, columns = predicted):")
print(cm_nb)

In [ ]:
# Top-10 informative words: a word must appear at least 20 times in each class (otherwise rare
# words that never occur in one class dominate the ranking when smoothing is off)
vocab = np.array(bow.get_feature_names_out())
y_tr = train["label"].values
cp = np.asarray(X_tr_bow[y_tr == "positive"].sum(axis=0)).ravel()
cn = np.asarray(X_tr_bow[y_tr == "negative"].sum(axis=0)).ravel()
ok = (cp >= 20) & (cn >= 20)
with np.errstate(divide="ignore", invalid="ignore"):
    score = np.log((cp / cp.sum()) / (cn / cn.sum()))
score = np.where(ok, score, np.nan)
top_pos = np.argsort(-np.nan_to_num(score, nan=-1e9))[:10]
top_neg = np.argsort(np.nan_to_num(score, nan=1e9))[:10]
top_words = pd.DataFrame({
    "predicts positive": vocab[top_pos],
    "predicts negative": vocab[top_neg],
})
top_words

In [ ]:
# Top-10 informative TF-IDF features (same log-ratio method, applied to class-total TF-IDF weight;
# each term must appear in at least 20 training reviews of each class). Bigrams are included.
vocab_tf = np.array(tfidf.get_feature_names_out())
y_tr = train["label"].values
pos_m, neg_m = (y_tr == "positive"), (y_tr == "negative")
wp = np.asarray(X_tr_tfidf[pos_m].sum(axis=0)).ravel(); wn = np.asarray(X_tr_tfidf[neg_m].sum(axis=0)).ravel()
dp = np.asarray((X_tr_tfidf[pos_m] > 0).sum(axis=0)).ravel(); dn = np.asarray((X_tr_tfidf[neg_m] > 0).sum(axis=0)).ravel()
ok = (dp >= 20) & (dn >= 20)
with np.errstate(divide="ignore", invalid="ignore"):
    score_tf = np.log((wp / wp.sum()) / (wn / wn.sum()))
score_tf = np.where(ok, score_tf, np.nan)
tp_pos = np.argsort(-np.nan_to_num(score_tf, nan=-1e9))[:10]
tp_neg = np.argsort(np.nan_to_num(score_tf, nan=1e9))[:10]
top_words_tfidf = pd.DataFrame({
    "predicts positive": vocab_tf[tp_pos],
    "predicts negative": vocab_tf[tp_neg],
})
top_words_tfidf

In [ ]:
# 2-D view of the TF-IDF space (truncated SVD = PCA without centering, appropriate for sparse input)
svd = TruncatedSVD(n_components=2, random_state=SEED).fit(X_tr_tfidf)
rng = np.random.RandomState(SEED)
pick = rng.choice(X_te_tfidf.shape[0], size=min(2000, X_te_tfidf.shape[0]), replace=False)
emb = svd.transform(X_te_tfidf[pick]); labs = test["label"].values[pick]
fig, ax = plt.subplots(figsize=(6, 4.5))
colors = {"negative": "#b91c1c", "neutral": "#9ca3af", "positive": "#2b6ca3"}
for c in LABELS:
    m = labs == c
    ax.scatter(emb[m, 0], emb[m, 1], s=6, alpha=0.5, c=colors[c], label=c)
ax.set_xlabel("component 1"); ax.set_ylabel("component 2"); ax.legend(markerscale=2)
ax.set_title("TF-IDF space, 2-D projection (2,000 test reviews)")
plt.tight_layout()
fig.savefig(FIG_DIR / "fig_tfidf_2d.png", dpi=200)
plt.show()

In [ ]:
# Error analysis
test = test.copy()
test["pred"] = pred_nb
test["n_words"] = test["doc"].str.split().str.len()
bins = [(0, 19, "short (<20 words)"), (20, 79, "medium (20-79)"), (80, 10**6, "long (80+)")]
rows = []
for lo, hi, name in bins:
    sub = test[(test["n_words"] >= lo) & (test["n_words"] <= hi)]
    rows.append({"slice": name, "n": len(sub), "error rate (%)": round((sub["pred"] != sub["label"]).mean() * 100, 1)})
for c in LABELS:
    sub = test[test["label"] == c]
    rows.append({"slice": f"true {c}", "n": len(sub), "error rate (%)": round((sub["pred"] != c).mean() * 100, 1)})
display(pd.DataFrame(rows))

errors = test[test["pred"] != test["label"]].sample(n=10, random_state=SEED)[["label", "pred", "n_words", "doc"]]
errors["doc"] = errors["doc"].str.slice(0, 160)
errors.to_csv(ROOT / "report" / "misclassified_sample.csv", index=False)
errors

In [ ]:
# Export one complete scikit-learn Pipeline for the Streamlit app (repo contract: models/README.md).
# It takes raw strings and contains all text preprocessing. Input = Summary + " " + Text, the same
# composition used for training. The app must join the two fields the same way before calling predict().
from sklearn.pipeline import Pipeline
pipe = Pipeline([
    ("bow", CountVectorizer(max_features=20000, lowercase=True)),
    ("nb", MultinomialNB(alpha=1e-9)),
])
pipe.fit(train["doc"], train["label"])  # training split only
assert (pipe.predict(test["doc"]) == pred_nb).all(), "pipeline must reproduce the baseline predictions"
assert list(pipe.classes_) == LABELS
joblib.dump(pipe, MODEL_DIR / "naive_bayes.joblib")
print("Saved:", sorted(p.name for p in MODEL_DIR.glob("*.joblib")))